### What are Missing Values?

Missing values are empty spots in the dataset where data should have 
been recorded but wasn't. In pandas these usually show up as `NaN`.

Example: A customer's TotalCharges might be missing if they just signed 
up and no bill has been generated yet.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df.isnull().sum())

CustomerID            0
Age                  80
Gender                0
TenureYears           0
MonthlyCharges        0
TotalCharges        151
ContractType          0
PaymentMethod         0
InternetService       0
TechSupport           0
SeniorCitizen         0
Partner               0
PaperlessBilling      0
Churn                 0
dtype: int64


### Why Missing Values Occur

Common real-world reasons:
- Data entry mistakes (someone forgot to fill a field).
- A value genuinely doesn't apply (e.g. no internet service, so no 
  internet-related charge).
- System/technical errors during data collection.
- The value wasn't available yet at the time of recording.

In [2]:
# Percentage of missing values per column
missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent[missing_percent > 0])

Age             1.581028
TotalCharges    2.984190
dtype: float64


### MCAR - Missing Completely At Random

The missing value has no relationship at all with any other data - 
it's pure chance. 

Example: A random system glitch drops a few values with no pattern - 
these missing "Age" values don't depend on income, gender, or anything 
else in the data.

In [3]:
# Check if missing Age values are spread randomly across ContractType groups
missing_age = df[df["Age"].isnull()]
print(missing_age["ContractType"].value_counts(normalize=True))
print(df["ContractType"].value_counts(normalize=True))

ContractType
Two Year          0.3500
Month-to-Month    0.3375
One Year          0.3125
Name: proportion, dtype: float64
ContractType
Two Year          0.337945
Month-to-Month    0.337352
One Year          0.324704
Name: proportion, dtype: float64


### MAR - Missing At Random

The missing value is related to another observed column, but not to the 
missing value itself.

Example: Customers with "Month-to-Month" contracts might be more likely 
to have missing TotalCharges, because they are newer customers who joined 
recently.

In [4]:
# Check if missing TotalCharges is linked to ContractType
missing_total = df[df["TotalCharges"].isnull()]
print(missing_total["ContractType"].value_counts(normalize=True))
print(df["ContractType"].value_counts(normalize=True))

ContractType
Two Year          0.397351
Month-to-Month    0.337748
One Year          0.264901
Name: proportion, dtype: float64
ContractType
Two Year          0.337945
Month-to-Month    0.337352
One Year          0.324704
Name: proportion, dtype: float64


### MNAR - Missing Not At Random

The missing value is related to the value itself, which wasn't recorded.

Example: Customers with very high MonthlyCharges might deliberately skip 
providing certain details, or high earners often don't disclose income - 
the reason for missing is tied to the value that's missing.

In [5]:
# Illustration only - compare MonthlyCharges for rows where Age is missing vs not
print("Avg MonthlyCharges when Age missing:", df[df["Age"].isnull()]["MonthlyCharges"].mean())
print("Avg MonthlyCharges when Age present:", df[df["Age"].notnull()]["MonthlyCharges"].mean())

Avg MonthlyCharges when Age missing: 97.86683498691767
Avg MonthlyCharges when Age present: 87.17325889021511


### Detecting Missing Values

We use `isnull()` or `isna()` to find missing values, and `sum()` to 
count them per column.

In [6]:
print(df.isnull().sum())
print("\nTotal missing values in dataset:", df.isnull().sum().sum())

CustomerID            0
Age                  80
Gender                0
TenureYears           0
MonthlyCharges        0
TotalCharges        151
ContractType          0
PaymentMethod         0
InternetService       0
TechSupport           0
SeniorCitizen         0
Partner               0
PaperlessBilling      0
Churn                 0
dtype: int64

Total missing values in dataset: 231


### Missing Value Percentage

Knowing the count isn't enough - the percentage tells us how serious the 
problem is. A column with 2% missing is very different from one with 60% 
missing.

In [7]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": missing_percent
})
print(missing_summary[missing_summary["Missing Count"] > 0])

              Missing Count  Missing %
Age                      80   1.581028
TotalCharges            151   2.984190


### Dropping Rows

Removing rows that contain missing values.

**When to use:** Missing percentage is very small (e.g. under 2-3%) and 
losing those rows won't hurt the dataset.
**When not to use:** Missing percentage is high - you'd lose too much 
useful data.
**Advantage:** Simple, no assumptions made.
**Limitation:** Reduces dataset size, can lose valuable information.

In [8]:
df_drop_rows = df.dropna(subset=["Age"])
print("Original rows:", len(df), "| After dropping missing Age rows:", len(df_drop_rows))

Original rows: 5060 | After dropping missing Age rows: 4980


### Dropping Columns

Removing an entire column if too much of it is missing.

**When to use:** A column has extremely high missing percentage (e.g. 
above 50-60%) and isn't critical.
**When not to use:** The column is important for the business or model, 
even with missing values.
**Advantage:** Removes a mostly-useless column completely.
**Limitation:** We lose that feature entirely, even the data that was 
present.

In [9]:
threshold = 50  # percent
cols_to_check = missing_percent[missing_percent > threshold].index.tolist()
print("Columns that would be dropped (none currently):", cols_to_check)

Columns that would be dropped (none currently): []


### Mean Imputation

Filling missing numeric values with the column's average.

**When to use:** Data is roughly normally distributed with no big 
outliers.
**When not to use:** Data is skewed or has outliers - mean gets pulled 
away from the "typical" value.
**Advantage:** Simple, keeps all rows.
**Limitation:** Reduces variance, can distort the distribution.

In [10]:
df_mean = df.copy()
mean_age = df_mean["Age"].mean()
df_mean["Age"] = df_mean["Age"].fillna(mean_age)
print("Mean used for imputation:", round(mean_age, 2))
print("Missing Age after imputation:", df_mean["Age"].isnull().sum())

Mean used for imputation: 43.76
Missing Age after imputation: 0


### Median Imputation

Filling missing numeric values with the column's median (middle value).

**When to use:** Data is skewed or has outliers - median is less 
affected by extreme values than mean.
**When not to use:** Data is clean and normally distributed (mean is 
fine then).
**Advantage:** Robust to outliers.
**Limitation:** Still just one fixed value - doesn't capture real 
variation.

In [11]:
df_median = df.copy()
median_total = df_median["TotalCharges"].median()
df_median["TotalCharges"] = df_median["TotalCharges"].fillna(median_total)
print("Median used for imputation:", round(median_total, 2))
print("Missing TotalCharges after imputation:", df_median["TotalCharges"].isnull().sum())

Median used for imputation: 4564.54
Missing TotalCharges after imputation: 0


### Mode Imputation

Filling missing categorical values with the most frequent value in the 
column.

**When to use:** Categorical column with one dominant/common category.
**When not to use:** Categories are fairly evenly distributed - mode 
imputation would create a false pattern.
**Advantage:** Keeps the column categorical, simple to apply.
**Limitation:** Can artificially boost the most common category.

In [12]:
df_mode = df.copy()
mode_contract = df_mode["ContractType"].mode()[0]
df_mode["ContractType"] = df_mode["ContractType"].fillna(mode_contract)
print("Mode used:", mode_contract)

Mode used: Two Year


### Constant Value Imputation

Filling missing values with a fixed value chosen by us, like "Unknown" 
or 0.

**When to use:** Missing itself carries meaning (e.g. "Unknown" is a 
valid business category).
**When not to use:** The constant doesn't reflect reality and would 
mislead the model.
**Advantage:** Preserves the fact that data was missing, easy to trace.
**Limitation:** Can introduce a fake category/value if used carelessly.

In [13]:
df_constant = df.copy()
df_constant["ContractType"] = df_constant["ContractType"].fillna("Unknown")
print(df_constant["ContractType"].value_counts())

ContractType
Two Year          1710
Month-to-Month    1707
One Year          1643
Name: count, dtype: int64


### Forward Fill

Filling a missing value using the previous row's value.

**When to use:** Data has a natural order (like time series) and values 
change slowly.
**When not to use:** Rows are independent of each other (like customer 
records with no order) - forward fill would make no logical sense.
**Advantage:** Good for sequential/time-based data.
**Limitation:** Wrong choice for unordered data.

In [14]:
df_ffill = df.sort_values("JoinDate").copy() if "JoinDate" in df.columns else df.copy()
df_ffill["Age"] = df_ffill["Age"].ffill()
print("Missing Age after forward fill:", df_ffill["Age"].isnull().sum())

Missing Age after forward fill: 0


### Backward Fill

Filling a missing value using the next row's value.

**When to use:** Same as forward fill - sequential/time-based data, but 
filling from the future value makes sense for that context.
**When not to use:** Unordered/independent data.
**Advantage:** Useful when the next value is a better estimate than the 
previous one.
**Limitation:** Can "leak" future information if not used carefully.

In [15]:
df_bfill = df.sort_values("JoinDate").copy() if "JoinDate" in df.columns else df.copy()
df_bfill["Age"] = df_bfill["Age"].bfill()
print("Missing Age after backward fill:", df_bfill["Age"].isnull().sum())

Missing Age after backward fill: 0


### Interpolation

Estimating a missing value based on the values around it, usually by 
drawing a straight line between the known points.

**When to use:** Numeric, ordered data (like time series) where values 
change gradually.
**When not to use:** Data has no order, or values jump around 
unpredictably.
**Advantage:** More accurate than a single fixed fill value for trending 
data.
**Limitation:** Not meaningful for categorical or unordered data.

In [16]:
df_interp = df.sort_values("JoinDate").copy() if "JoinDate" in df.columns else df.copy()
df_interp["TotalCharges"] = df_interp["TotalCharges"].interpolate()
print("Missing TotalCharges after interpolation:", df_interp["TotalCharges"].isnull().sum())

Missing TotalCharges after interpolation: 0


### Group-Based Imputation

Filling missing values using statistics calculated within a related 
group, instead of the whole dataset.

**When to use:** The column's typical value differs a lot between 
groups (e.g. MonthlyCharges differs a lot by ContractType).
**When not to use:** No meaningful groups exist, or group sizes are too 
small to be reliable.
**Advantage:** More accurate than a single overall mean/median.
**Limitation:** More complex, needs a good grouping column.

In [17]:
df_group = df.copy()
df_group["TotalCharges"] = df_group.groupby("ContractType")["TotalCharges"] \
    .transform(lambda x: x.fillna(x.median()))
print("Missing TotalCharges after group-based imputation:", df_group["TotalCharges"].isnull().sum())

Missing TotalCharges after group-based imputation: 0


### KNN Imputation

Fills a missing value by looking at the "K" most similar rows (nearest 
neighbours) based on other columns, and using their average.

**When to use:** Data has clear relationships between numeric features, 
and dataset isn't too large.
**When not to use:** Dataset is very large (slow) or has too many 
missing values.
**Advantage:** Considers relationships between features, often more 
accurate.
**Limitation:** Computationally expensive, sensitive to feature scaling.

In [18]:
from sklearn.impute import KNNImputer

df_knn = df.copy()
numeric_cols = ["Age", "TenureYears", "MonthlyCharges", "TotalCharges"]

imputer = KNNImputer(n_neighbors=5)
df_knn[numeric_cols] = imputer.fit_transform(df_knn[numeric_cols])

print("Missing values after KNN imputation:\n", df_knn[numeric_cols].isnull().sum())

Missing values after KNN imputation:
 Age               0
TenureYears       0
MonthlyCharges    0
TotalCharges      0
dtype: int64


### Iterative Imputation

Predicts each missing value using a model trained on the other columns, 
repeating this process multiple times until values stabilize.

**When to use:** Multiple columns have missing values with real 
relationships between them, and accuracy matters more than speed.
**When not to use:** Simple/small datasets where basic imputation is 
good enough - this method is slower and more complex.
**Advantage:** Often the most accurate imputation method.
**Limitation:** Slow, more complex to explain, can overfit patterns.

In [19]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df_iter = df.copy()
imputer = IterativeImputer(random_state=42)
df_iter[numeric_cols] = imputer.fit_transform(df_iter[numeric_cols])

print("Missing values after iterative imputation:\n", df_iter[numeric_cols].isnull().sum())

Missing values after iterative imputation:
 Age               0
TenureYears       0
MonthlyCharges    0
TotalCharges      0
dtype: int64


### Why Not Simply Fill Every Missing Value With the Mean

Using mean for everything is tempting because it's easy, but it's often 
wrong:
- For skewed data, mean doesn't represent the "typical" value - median 
  is better.
- For categorical columns, mean doesn't even make sense - mode is 
  needed.
- Mean imputation reduces variance and can hide real patterns in the 
  data, which affects the model's ability to learn properly.
- Sometimes missing itself is meaningful, and filling with mean removes 
  that signal.

The right technique depends on the column type, the reason for 
missingness (MCAR/MAR/MNAR), and how much data is missing.

In [20]:
# Compare mean vs median for a skewed column to show the difference
print("Mean of MonthlyCharges:", df["MonthlyCharges"].mean())
print("Median of MonthlyCharges:", df["MonthlyCharges"].median())

Mean of MonthlyCharges: 87.34232728700094
Median of MonthlyCharges: 84.56
